In [50]:
import pandas as pd
import geopandas as gpd

In [51]:
area_path = "../../results/spatial/iris.parquet"
housing_path = "../../results/parking/housing.parquet"
network_path = "../../results/parking/network.parquet"

parking_length = 5.0

output_path = "../../results/parking/parking_pressure.parquet"

In [52]:
df_area = gpd.read_parquet(area_path)
df_housing = pd.read_parquet(housing_path)
df_network = pd.read_parquet(network_path)

In [53]:
df_housing

,iris,has_minimum_one_parking,has_minimum_one_car,has_minimum_two_cars
0,010010000,265.393980,102.803725,193.712822
1,010020000,79.127572,30.193416,71.839506
2,010040101,532.830267,433.883533,238.594985
3,010040102,1194.036317,1059.946678,409.252950
4,010040201,1307.809318,1109.955313,518.769357
...,...,...,...,...
35350,693890502,720.345580,717.834256,204.468833
35351,693890604,267.301727,383.852756,95.351620
35352,693890605,210.037971,266.453195,43.833064
35353,693890606,321.883791,518.757010,157.452259


In [54]:
df_housing

,iris,has_minimum_one_parking,has_minimum_one_car,has_minimum_two_cars
0,010010000,265.393980,102.803725,193.712822
1,010020000,79.127572,30.193416,71.839506
2,010040101,532.830267,433.883533,238.594985
3,010040102,1194.036317,1059.946678,409.252950
4,010040201,1307.809318,1109.955313,518.769357
...,...,...,...,...
35350,693890502,720.345580,717.834256,204.468833
35351,693890604,267.301727,383.852756,95.351620
35352,693890605,210.037971,266.453195,43.833064
35353,693890606,321.883791,518.757010,157.452259


In [55]:
df_parking = df_area.copy()
df_parking = pd.merge(df_parking, df_housing, how = "left", on = "iris")
df_parking = pd.merge(df_parking, df_network, how = "left", on = "iris")

In [56]:
df_parking["has_minimum_one_car"] = df_parking["has_minimum_one_car"].fillna(0.0)
df_parking["has_minimum_two_cars"] = df_parking["has_minimum_two_cars"].fillna(0.0)
df_parking["has_minimum_one_parking"] = df_parking["has_minimum_one_parking"].fillna(1.0)

In [57]:
df_parking["parking_pressure"] = df_parking["has_minimum_one_car"] / (
    df_parking["has_minimum_one_parking"] + df_parking["length"] / parking_length)

df_pressure = df_parking[["iris", "parking_pressure", "geometry"]]
df_pressure.to_parquet(output_path)

In [58]:
df_pressure["parking_pressure"].describe()

count    2472.000000
mean        0.108333
std         0.170674
min         0.000000
25%         0.006188
50%         0.017266
75%         0.148725
max         1.115432
Name: parking_pressure, dtype: float64